# Recurrent Neural Network (LSTM)

Prepares sequence data grouped by session with sliding windows, builds LSTM per spec, trains with early stopping and LR reduction, evaluates, visualizes temporal patterns, compares RNN variants, experiments with sequence length, and saves model to H5 and TFLite with metadata.

In [ ]:

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
from IPython.display import display
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

try:
    import tensorflow as tf
except ImportError:
    raise ImportError("TensorFlow not installed. Please: pip install tensorflow")
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
np.random.seed(42)
try:
    tf.random.set_seed(42)
except Exception:
    pass

from schema_utils import detect_feature_columns, infer_session_sort_columns, create_sequences


In [ ]:
# 2. Data Loading - use train/test CSVs
import os
import pandas as pd
DATA_DIR = os.path.join('..','datasets')
TRAIN_PATH = os.path.join(DATA_DIR, 'training_data_train.csv')
TEST_PATH = os.path.join(DATA_DIR, 'training_data_test.csv')
if not os.path.exists(TRAIN_PATH) or not os.path.exists(TEST_PATH):
    raise FileNotFoundError("Missing train/test CSVs in ../datasets: training_data_train.csv and training_data_test.csv")
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
from schema_utils import detect_feature_columns
schema = detect_feature_columns(df_train)
if not schema['label_col']:
    raise ValueError('No label column detected. Expected one of: label/class/target')
label_col = schema['label_col']
print('Train size:', len(df_train), 'Test size:', len(df_test))
print('Train balance:
', df_train[label_col].value_counts(normalize=True))
print('Test balance:
', df_test[label_col].value_counts(normalize=True))

# Sequence setup
from schema_utils import infer_session_sort_columns, create_sequences
feature_cols = schema['zone_cols']
session_col_train, sort_col_train = infer_session_sort_columns(df_train)
session_col_test, sort_col_test = infer_session_sort_columns(df_test)
SEQ_LEN = 10



In [ ]:
# Build sequences separately for train and test
X_train_sequences, y_str_train = create_sequences(df_train, SEQ_LEN, feature_cols, label_col, session_col_train, sort_col_train)
X_test_sequences, y_str_test = create_sequences(df_test, SEQ_LEN, feature_cols, label_col, session_col_test, sort_col_test)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_str_train)
y_test = le.transform(y_str_test)
print('Train sequences:', X_train_sequences.shape, 'Test sequences:', X_test_sequences.shape)
print('Classes:', list(le.classes_))



In [ ]:
# Build sequences separately for train and test
X_train_sequences, y_str_train = create_sequences(df_train, SEQ_LEN, feature_cols, label_col, session_col_train, sort_col_train)
X_test_sequences, y_str_test = create_sequences(df_test, SEQ_LEN, feature_cols, label_col, session_col_test, sort_col_test)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_str_train)
y_test = le.transform(y_str_test)
print('Train sequences:', X_train_sequences.shape, 'Test sequences:', X_test_sequences.shape)
print('Classes:', list(le.classes_))



In [ ]:

# Build LSTM model per spec
model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, len(feature_cols))),
    layers.LSTM(32, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
    layers.LSTM(16, return_sequences=False, dropout=0.2),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)

# Train
start_time = time.time()
history = model.fit(
    X_train_sequences, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)
training_time = time.time() - start_time
print(f'Training completed in {training_time:.2f} seconds. Epochs:', len(history.history['loss']))

# Plot training history
hist_df = pd.DataFrame(history.history)
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(hist_df['accuracy'], label='train acc')
plt.plot(hist_df['val_accuracy'], label='val acc')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend()
plt.subplot(1,2,2)
plt.plot(hist_df['loss'], label='train loss')
plt.plot(hist_df['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:

# Evaluate on test
test_loss, test_accuracy = model.evaluate(X_test_sequences, y_test, verbose=0)
print('Test loss:', test_loss)
print('Test accuracy:', test_accuracy)

# Predict probabilities and classes
probs = model.predict(X_test_sequences[:20], verbose=0)
preds = np.argmax(model.predict(X_test_sequences, verbose=0), axis=1)
print('Probabilities (first 20 sequences):')
print(pd.DataFrame(probs, columns=le.classes_))
print('Predicted labels (first 20):', le.inverse_transform(np.argmax(probs, axis=1)))

# Confusion matrix and classification report
cm = confusion_matrix(y_test, preds)
print('Confusion matrix:
', cm)
print(classification_report(y_test, preds, target_names=list(le.classes_)))
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix'); plt.tight_layout(); plt.show()


In [ ]:

# Analyze temporal patterns for example sequences

def pick_example_sequences(X, y, target_int, n=1):
    idxs = np.where(y == target_int)[0]
    if len(idxs) == 0:
        return []
    return [X[i] for i in idxs[:n]]

targets = list(le.classes_)
examples = {}
for target in targets:
    t_int = le.transform([target])[0]
    seqs = pick_example_sequences(X_test_sequences, y_test, t_int, n=1)
    if seqs:
        examples[target] = seqs[0]

for label, seq in examples.items():
    # seq shape: (SEQ_LEN, 12)
    plt.figure(figsize=(10,4))
    sns.heatmap(seq.T, cmap='magma', cbar=True, xticklabels=range(SEQ_LEN), yticklabels=feature_cols)
    plt.title(f'Feature heatmap over {SEQ_LEN} timesteps ({label})')
    plt.xlabel('Timestep'); plt.ylabel('Feature'); plt.tight_layout(); plt.show()

    # Sliding predictions across the sequence when possible
    # For each step k, predict using the window ending at k (requires prior steps);
    # Here we approximate by using the fixed window (seq) and reporting the final prediction.
    prob = model.predict(seq.reshape(1, SEQ_LEN, len(feature_cols)), verbose=0)[0]
    plt.figure(figsize=(6,3))
    plt.bar(targets, prob)
    plt.title(f'Prediction probabilities for sequence ({label})')
    plt.ylabel('Probability'); plt.tight_layout(); plt.show()


In [ ]:

# Compare RNN variants: SimpleRNN, LSTM, GRU, Bidirectional LSTM

def build_simple_rnn():
    m = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, len(feature_cols))),
        layers.SimpleRNN(32, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        layers.SimpleRNN(16, return_sequences=False, dropout=0.2),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(3, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_lstm():
    m = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, len(feature_cols))),
        layers.LSTM(32, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        layers.LSTM(16, return_sequences=False, dropout=0.2),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(3, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_gru():
    m = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, len(feature_cols))),
        layers.GRU(32, return_sequences=True, dropout=0.2, recurrent_dropout=0.2),
        layers.GRU(16, return_sequences=False, dropout=0.2),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(3, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def build_bilstm():
    m = keras.Sequential([
        layers.Input(shape=(SEQ_LEN, len(feature_cols))),
        layers.Bidirectional(layers.LSTM(32, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
        layers.Bidirectional(layers.LSTM(16, return_sequences=False, dropout=0.2)),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(3, activation='softmax')
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

variants = [
    ('SimpleRNN', build_simple_rnn),
    ('LSTM', build_lstm),
    ('GRU', build_gru),
    ('BiLSTM', build_bilstm)
]

results = []
for name, builder in variants:
    m = builder()
    start = time.time()
    h = m.fit(X_train_sequences, y_train, epochs=100, batch_size=64, validation_split=0.2,
              callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
                         ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5)],
              verbose=0)
    train_time = time.time() - start
    acc = m.evaluate(X_test_sequences, y_test, verbose=0)[1]
    params = m.count_params()
    results.append({'variant': name, 'test_accuracy': acc, 'training_time_s': train_time, 'params': params, 'epochs': len(h.history['loss'])})

var_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False)
display(var_df)
plt.figure(figsize=(8,4)); sns.barplot(x='variant', y='test_accuracy', data=var_df)
plt.title('RNN Variant Comparison (Accuracy)'); plt.tight_layout(); plt.show()


In [ ]:
# Build sequences separately for train and test
X_train_sequences, y_str_train = create_sequences(df_train, SEQ_LEN, feature_cols, label_col, session_col_train, sort_col_train)
X_test_sequences, y_str_test = create_sequences(df_test, SEQ_LEN, feature_cols, label_col, session_col_test, sort_col_test)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_train = le.fit_transform(y_str_train)
y_test = le.transform(y_str_test)
print('Train sequences:', X_train_sequences.shape, 'Test sequences:', X_test_sequences.shape)
print('Classes:', list(le.classes_))



In [ ]:

# Save base LSTM model and TFLite
h5_path = os.path.join('..','models','lstm_model.h5')
model.save(h5_path)
print('Saved H5 model to', h5_path)

# Try TFLite conversion (may be limited for LSTM)
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    with open(os.path.join('..','models','lstm_model.tflite'), 'wb') as f:
        f.write(tflite_model)
    print('Saved TFLite model as lstm_model.tflite')
except Exception as e:
    print('TFLite conversion failed or partially supported:', e)

# Save metadata
meta = {
    'sequence_length': SEQ_LEN,
    'lstm_units': [32, 16],
    'test_accuracy': float('nan'),  # will fill after eval if desired
    'training_time_s': float('nan'),
    'feature_columns': feature_cols
}
# Attempt to fill from variables if present
try:
    meta['test_accuracy'] = float(test_accuracy)
    meta['training_time_s'] = float(training_time)
except Exception:
    pass
with open(os.path.join('..','models','lstm_model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved metadata json: lstm_model_meta.json')
